# Research 02 — XGBoost diversity blend

**Hypothesis:** XGBoost native categorical trees score below the CatBoost/LightGBM blend alone, but may contribute useful error diversity. Test one predeclared 80/20 probability blend; no hyperparameter or blend-weight sweep. Use the same three stratified folds (seed 42) and saved OOF blend probabilities. Test is accessed only if the candidate improves OOF macro F1.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv")
y = train.segment.to_numpy(dtype=int)
base = pd.read_csv(ROOT / "predictions/segment-oof.csv")
assert len(base) == len(train) and np.array_equal(base.segment.to_numpy(), y)
p_base = base[[f"prob_{i}" for i in range(4)]].to_numpy()
X = train.drop(columns=["segment", "player_id"]).copy()
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype("category")
print(f"Rows={len(train):,}; native categorical features={len(cat_cols)}")

Rows=104,000; native categorical features=10


## Fixed XGBoost configuration

This exact configuration is screened as a standalone alternative and as one fixed 20% component in the existing OOF blend. It is not tuned on Test.

In [2]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
p_xgb = np.zeros((len(train), 4), dtype=float)
for fold, (fit_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=4,
        max_depth=6,
        learning_rate=0.07,
        n_estimators=400,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=5,
        tree_method="hist",
        enable_categorical=True,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=4,
        verbosity=0,
    )
    model.fit(X.iloc[fit_idx], y[fit_idx])
    p_xgb[valid_idx] = model.predict_proba(X.iloc[valid_idx])
    print(
        f"Fold {fold}: macro F1={f1_score(y[valid_idx], p_xgb[valid_idx].argmax(1), average='macro'):.5f}",
        flush=True,
    )

p_blend = 0.8 * p_base + 0.2 * p_xgb


def metric_row(name, probabilities):
    pred = probabilities.argmax(axis=1)
    return {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }


results = pd.DataFrame(
    [
        metric_row("CatBoost + LightGBM (existing)", p_base),
        metric_row("80% existing + 20% XGBoost", p_blend),
        metric_row("XGBoost", p_xgb),
    ]
)
display(results)
results.to_csv(ROOT / "assets/segment-xgboost-diversity-results.csv", index=False)
pd.DataFrame(p_xgb, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions/segment-xgboost-oof.csv", index=False
)
pd.DataFrame(p_blend, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions/segment-xgboost-blend-oof.csv", index=False
)
promote = (
    results.loc[results.candidate == "80% existing + 20% XGBoost", "macro_f1"].iloc[0]
    > results.loc[results.candidate == "CatBoost + LightGBM (existing)", "macro_f1"].iloc[0]
)
print("OOF promotion decision:", promote)

Fold 1: macro F1=0.74593


Fold 2: macro F1=0.74613


Fold 3: macro F1=0.74913


,candidate,macro_f1,weighted_f1
0,CatBoost + LightGBM (existing),0.750895,0.764683
1,80% existing + 20% XGBoost,0.750769,0.764667
2,XGBoost,0.747070,0.761152


OOF promotion decision: False


## Prediction-only test pass if validation improves

When the fixed blend beats the saved OOF candidate, refit XGBoost on all labeled rows and reuse the already full-data CatBoost/LightGBM checkpoints to produce blend probabilities. The test features are never used to fit or tune any component. The competition submission file is not modified.

In [3]:
if promote:
    test = pd.read_csv(ROOT / "data/test.csv")
    test_x = test.drop(columns=["player_id"], errors="ignore").copy()
    for col in cat_cols:
        test_x[col] = pd.Categorical(test_x[col], categories=X[col].cat.categories)
    final_xgb = XGBClassifier(
        objective="multi:softprob",
        num_class=4,
        max_depth=6,
        learning_rate=0.07,
        n_estimators=400,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=5,
        tree_method="hist",
        enable_categorical=True,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=4,
        verbosity=0,
    ).fit(X, y)
    p_xgb_test = final_xgb.predict_proba(test_x)

    # Recreate the existing full-data CatBoost/LightGBM probabilities.
    cat_cols_raw = (
        train.drop(columns=["segment", "player_id"])
        .select_dtypes(include=["object", "string"])
        .columns.tolist()
    )
    raw_test = test.drop(columns=["player_id"], errors="ignore").copy()
    raw_test[cat_cols_raw] = raw_test[cat_cols_raw].fillna("__MISSING__").astype(str)
    full_cat = CatBoostClassifier()
    full_cat.load_model(str(ROOT / "models/segment-catboost.cbm"))
    p_cat = full_cat.predict_proba(raw_test)

    def engineered_features(frame):
        x = frame.drop(
            columns=[
                "segment",
                "player_id",
                "random_metric_1",
                "random_metric_2",
                "random_metric_3",
            ],
            errors="ignore",
        ).copy()
        ratios = {
            "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
            "spend_per_account_day": ("total_spending_thb", "account_age_days"),
            "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
            "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
            "gifts_per_friend": ("gifts_sent_received", "friend_count"),
            "invites_per_friend": ("friend_invites_sent", "friend_count"),
            "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
        }
        for name, (a, b) in ratios.items():
            x[name] = x[a] / (x[b].abs() + 1)
        x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
        x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
        x["missing_count"] = x.isna().sum(axis=1)
        return x.replace([np.inf, -np.inf], np.nan)

    lgb_bundle = joblib.load(ROOT / "models/segment-lightgbm.joblib")
    lgb_test = engineered_features(test)
    for col, dtype in lgb_bundle["categorical_dtypes"].items():
        lgb_test[col] = lgb_test[col].astype(dtype)
    p_lgb = lgb_bundle["model"].predict_proba(lgb_test)
    p_existing_test = (p_cat + p_lgb) / 2
    p_final = 0.8 * p_existing_test + 0.2 * p_xgb_test
    output = pd.DataFrame({"player_id": test.player_id, "segment": p_final.argmax(axis=1)})
    assert len(output) == 25889 and output.segment.isin(range(4)).all()
    output.to_csv(ROOT / "predictions/segment-xgboost-diversity-test.csv", index=False)
    final_xgb.save_model(str(ROOT / "models/segment-xgboost.json"))
    display(output.head())
    print(f"Saved {len(output):,} test predictions keyed by player_id; submit/index.csv unchanged.")
else:
    print("No test prediction generated: OOF score did not beat the current candidate.")

No test prediction generated: OOF score did not beat the current candidate.


## Research decision

Retain the blend only when its fixed OOF macro F1 exceeds the current candidate. A lower standalone XGBoost score is not by itself evidence against an ensemble, but a lower evaluated blend is a reason to reject the added complexity. This is one bounded diversity check, not a search over blend weights.